# NPP analysis
Calculate annual 60–90°N land-integrated NPP (Pg C yr⁻¹) and map Exp1/Exp4 to GWL.
When adding a model, include an entry in `MODEL_CONFIGS` and follow the existing directory naming convention.

In [1]:
from pathlib import Path
import sys

# Find this project from the notebook's working directory.
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "utils" / "common.py").is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Project folder containing utils/common.py was not found.")
    PROJECT_ROOT = PROJECT_ROOT.parent
CODE_DIR = PROJECT_ROOT / "utils"
sys.path.insert(0, str(CODE_DIR))

from common import find_project_root, load_ec_land_area, remap_land_area_to_esm, map_annual_to_gwl
PROJECT_ROOT = find_project_root(PROJECT_ROOT)
from npp import annual_regional_npp
import xarray as xr

MODEL_CONFIGS = [
    {"name": "EC-Earth3-ESM-1", "family": "EC", "member": "r1i1p1f1"},
    {"name": "GISS-E2-1-G-CC2", "family": "NASA", "member": "r1i1p1f3"},
    {"name": "UKESM1-2-LL", "family": "UKESM", "member": "r1i1p1f1"},
]
EXPERIMENTS = ("Exp1", "Exp2", "Exp4")
GWL_LEGS = {"Exp1": "ramp-up", "Exp4": "ramp-down-2c"}
LAT_MIN, LAT_MAX = 60.0, 90.0

# Keep derived plot data outside the GitHub repository and raw inputs.
OUTPUT_DIR = PROJECT_ROOT.parent / f"{PROJECT_ROOT.name}_plot_data" / "npp"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
# use Exp1 sample coordinates to remap EC land area to each NPP grid.
ec_land_area = load_ec_land_area(PROJECT_ROOT)
land_areas = {}
for config in MODEL_CONFIGS:
    family = config["family"]
    sample_folder = PROJECT_ROOT / "Exp1_data" / family / f"Exp1_{family}_npp"
    sample_file = sorted(sample_folder.glob("*.nc"))[0]
    with xr.open_dataset(sample_file) as sample:
        land_areas[config["name"]] = remap_land_area_to_esm(
            ec_land_area, sample["lat"].load().values, sample["lon"].load().values
        )

In [3]:
# Save annual totals for Exp1/Exp2/Exp4, GWL series for Exp1/Exp4, and the Exp2 full-record mean for each model.
for config in MODEL_CONFIGS:
    model, family = config["name"], config["family"]
    annual = {}
    for experiment in EXPERIMENTS:
        default_folder = f"{experiment}_{family}_npp"
        folder_name = config.get("folder_overrides", {}).get(experiment, default_folder)
        folder = PROJECT_ROOT / f"{experiment}_data" / family / folder_name
        files = sorted(folder.glob("*.nc"))
        annual[experiment] = annual_regional_npp(files, land_areas[model], lat_min=LAT_MIN, lat_max=LAT_MAX)
        print(f"{model} {experiment}: {annual[experiment].sizes['year']} complete years; "
              f"mean={float(annual[experiment].mean('year')):.3f} Pg C yr⁻¹")

    mapped = {
        experiment: map_annual_to_gwl(annual[experiment], model, config["member"], leg, PROJECT_ROOT)
        for experiment, leg in GWL_LEGS.items()
    }
    plot_data = xr.Dataset()
    for experiment, series in annual.items():
        plot_data[f"{experiment.lower()}_annual"] = series.rename({"year": f"year_{experiment.lower()}"})
    for experiment, series in mapped.items():
        rename = {"gwl": f"gwl_{experiment.lower()}"}
        if "year" in series.coords:
            rename["year"] = f"source_year_{experiment.lower()}"
        plot_data[f"{experiment.lower()}_gwl"] = series.rename(rename)
    plot_data["exp2_full_record_mean"] = annual["Exp2"].mean("year")
    plot_data.attrs.update({
        "model": model,
        "dataset_family": family,
        "latitude_band": "60–90°N",
        "units": "Pg C yr-1",
        "description": "Annual regional NPP and GWL series for plotting; raw monthly inputs are not included.",
    })
    filename = model.replace("/", "_").replace(" ", "_") + "_npp_plot_data.nc"
    output_file = OUTPUT_DIR / filename
    plot_data.to_netcdf(output_file)
    print(f"Saved plot data: {output_file}")


EC-Earth3-ESM-1 Exp1: 350 complete years; mean=5.204 Pg C yr⁻¹
EC-Earth3-ESM-1 Exp2: 500 complete years; mean=4.146 Pg C yr⁻¹
EC-Earth3-ESM-1 Exp4: 120 complete years; mean=2.409 Pg C yr⁻¹
Saved plot data: /mnt/e/lund/NGEA52/hysteresis/tipmip-hysteresis-highlat-ecosystem_plot_data/npp/EC-Earth3-ESM-1_npp_plot_data.nc
GISS-E2-1-G-CC2 Exp1: 290 complete years; mean=2.621 Pg C yr⁻¹
GISS-E2-1-G-CC2 Exp2: 300 complete years; mean=2.442 Pg C yr⁻¹
GISS-E2-1-G-CC2 Exp4: 97 complete years; mean=1.894 Pg C yr⁻¹
Saved plot data: /mnt/e/lund/NGEA52/hysteresis/tipmip-hysteresis-highlat-ecosystem_plot_data/npp/GISS-E2-1-G-CC2_npp_plot_data.nc
UKESM1-2-LL Exp1: 290 complete years; mean=5.371 Pg C yr⁻¹
UKESM1-2-LL Exp2: 501 complete years; mean=4.972 Pg C yr⁻¹
UKESM1-2-LL Exp4: 134 complete years; mean=3.747 Pg C yr⁻¹
Saved plot data: /mnt/e/lund/NGEA52/hysteresis/tipmip-hysteresis-highlat-ecosystem_plot_data/npp/UKESM1-2-LL_npp_plot_data.nc
